In [18]:
import pandas as pd
import numpy as np

# ===== 1. 파일 로드 =====
pop = pd.read_excel('Data/[공통] 전체_인구통계_정제완료.xlsx')
df = pd.read_excel('Data/[의료]지역별_백신_완전접종률_2026.xlsx', skiprows=3)

# ===== 2. 백신 데이터 정제 =====
df = pd.read_excel('Data/[의료]지역별_백신_완전접종률_2026.xlsx', skiprows=3)
df.columns = ['시도명', '시군구명', '1세접종률', '2세접종률', '3세접종률']

# 시도명 ffill
df['시도명'] = df['시도명'].ffill()

# 소계 제거, 타겟 지역만
df = df[df['시군구명'] != '소계'].copy()

# 숫자 변환
for col in ['1세접종률', '2세접종률', '3세접종률']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3개 평균
df['예방접종_원값'] = df[['1세접종률', '2세접종률', '3세접종률']].mean(axis=1)

# 시군구명 정리 (경기도 구 포함된 경우 시만 추출)
import re
def clean_sigungu(name):
    m = re.match(r'(\S+시)\s+\S+구', str(name))
    return m.group(1) if m else name

df['시군구명_정리'] = df['시군구명'].apply(clean_sigungu)

# 시군구 단위로 평균 (구 단위 있는 시는 시 단위로 통합)
df_sg = df.groupby(['시도명', '시군구명_정리'])['예방접종_원값'].mean().reset_index()
df_sg.columns = ['시도명', '시군구명', '예방접종_원값']

print('백신 데이터 시군구 수:', len(df_sg))
print(df_sg.head(10).to_string())

백신 데이터 시군구 수: 66
   시도명  시군구명    예방접종_원값
0  경기도   가평군  87.666667
1  경기도   고양시  90.600000
2  경기도   과천시  91.566667
3  경기도   광명시  92.433333
4  경기도   광주시  92.266667
5  경기도   구리시  93.400000
6  경기도   군포시  92.466667
7  경기도   김포시  91.633333
8  경기도  남양주시  92.700000
9  경기도  동두천시  88.766667


c:\Users\asia\miniconda3\envs\jin_3.11\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\asia\miniconda3\envs\jin_3.11\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [19]:
# ===== 3. 인구 데이터 동 목록 =====
import re

TARGET = ['서울특별시', '경기도 성남시', '경기도 고양시', '경기도 부천시', '경기도 광명시',
          '경기도 하남시', '경기도 남양주시', '경기도 용인시', '경기도 김포시',
          '경기도 구리시', '경기도 안양시', '인천광역시']

def make_sgkey(row):
    sido, sigungu = row['시도명'], row['시군구명']
    if sido == '경기도':
        m = re.match(r'(\S+시)', sigungu)
        return '경기도 ' + m.group(1) if m else sido + ' ' + sigungu
    return sido + ' ' + sigungu

pop['시군구키'] = pop.apply(make_sgkey, axis=1)
pop_t = pop[pop['시군구키'].apply(lambda k: any(k.startswith(t) for t in TARGET))].copy()

# ===== 4. 백신 데이터 시군구키 생성 =====
def make_vaccine_sgkey(row):
    sido, sigungu = row['시도명'], row['시군구명']
    if sido == '경기도':
        return '경기도 ' + sigungu
    elif sido == '서울특별시':
        return '서울특별시 ' + sigungu
    elif sido == '인천광역시':
        return '인천광역시 ' + sigungu
    return sido + ' ' + sigungu

df_sg['시군구키'] = df_sg.apply(make_vaccine_sgkey, axis=1)

# ===== 5. 동별로 시군구 값 동일 적용 =====
pop_dong = pop_t[['시군구키', '읍면동명']].copy()
pop_dong.columns = ['시군구키', '행정동']

final = pd.merge(pop_dong, df_sg[['시군구키', '예방접종_원값']], on='시군구키', how='left')

# ===== 6. Min-Max 정규화 =====
vmin, vmax = final['예방접종_원값'].min(), final['예방접종_원값'].max()
final['예방접종_정규화'] = (final['예방접종_원값'] - vmin) / (vmax - vmin)

# 시도명, 시군구명 분리
final['시도명'] = final['시군구키'].apply(lambda x: x.split(' ')[0])
final['시군구명'] = final['시군구키'].apply(lambda x: ' '.join(x.split(' ')[1:]))
final = final[['시도명', '시군구명', '행정동', '예방접종_원값', '예방접종_정규화']]

print(f'총 동 수: {len(final)}')
print(f'원값 범위: {vmin:.2f} ~ {vmax:.2f}')
print(f'null 수: {final["예방접종_원값"].isna().sum()}')
print(final.sort_values('예방접종_정규화', ascending=False).head(10).to_string())

# ===== 7. 저장 =====
final.to_csv('[건강]예방접종_동별지수.csv', index=False, encoding='utf-8-sig')
print('저장 완료!')

총 동 수: 863
원값 범위: 81.43 ~ 93.67
null 수: 0
       시도명 시군구명    행정동    예방접종_원값  예방접종_정규화
556  인천광역시   서구   가좌4동  93.666667       1.0
549  인천광역시   서구   석남1동  93.666667       1.0
550  인천광역시   서구   석남2동  93.666667       1.0
545  인천광역시   서구   청라3동  93.666667       1.0
546  인천광역시   서구   가정1동  93.666667       1.0
547  인천광역시   서구   가정2동  93.666667       1.0
548  인천광역시   서구   가정3동  93.666667       1.0
541  인천광역시   서구  검암경서동  93.666667       1.0
542  인천광역시   서구    연희동  93.666667       1.0
543  인천광역시   서구   청라1동  93.666667       1.0
저장 완료!


C:\Users\asia\AppData\Local\Temp\ipykernel_25244\1907484019.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop['시군구키'] = pop.apply(make_sgkey, axis=1)
